<a href="https://colab.research.google.com/github/lmbernardo7520112/desafio_bairesdev_embedded_vision_project_LMB/blob/main/FER2013_MobileNetV2_EfficientNet_Pruning_%2B_Quantiza%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! ls /content/drive/MyDrive/Colab\ Notebooks/kaggle.json

'/content/drive/MyDrive/Colab Notebooks/kaggle.json'


In [ ]:
# Instalar Kaggle API se não tiver
!pip install -q kaggle

# Autenticar (faça upload do kaggle.json no Colab antes)
# /content/drive/MyDrive/Colab\ Notebooks/kaggle.json
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Colab\ Notebooks/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar dataset FER2013
!kaggle datasets download -d msambare/fer2013
!unzip -q fer2013.zip -d data/


Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
  0% 0.00/60.3M [00:00<?, ?B/s]
100% 60.3M/60.3M [00:00<00:00, 646MB/s]


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, DepthwiseConv2D, BatchNormalization, ReLU
from tensorflow.keras.layers import MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout

num_labels = 7   # emoções
width, height = 48, 48

def MobileNet_Light(input_shape=(width, height, 1), num_classes=num_labels):
    model = Sequential(name="MobileNet_Light")

    # Bloco inicial
    model.add(Conv2D(32, (3,3), strides=(2,2), padding='same',
                     input_shape=input_shape, use_bias=False))
    model.add(BatchNormalization())
    model.add(ReLU(6.))  # ReLU6 para compatibilidade com quantização

    # Blocos Depthwise Separable
    def depthwise_block(filters, stride):
        model.add(DepthwiseConv2D((3,3), strides=(stride,stride), padding='same', use_bias=False))
        model.add(BatchNormalization())
        model.add(ReLU(6.))
        model.add(Conv2D(filters, (1,1), strides=(1,1), padding='same', use_bias=False))
        model.add(BatchNormalization())
        model.add(ReLU(6.))

    depthwise_block(64, 1)
    depthwise_block(128, 2)
    depthwise_block(128, 1)
    depthwise_block(256, 2)
    depthwise_block(256, 1)
    depthwise_block(512, 2)

    # Pooling + Classificação
    model.add(GlobalAveragePooling2D())
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))

    return model

# Criar e compilar
model = MobileNet_Light()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "MobileNet_Light"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 24, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 24, 24, 32)     │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 12, 12, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 12, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 12, 12, 128)    │         1,152 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 276,615 (1.06 MB)

 Trainable params: 272,135 (1.04 MB)

 Non-trainable params: 4,480 (17.50 KB)

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Agora aumentamos a resolução para 96x96 (MobileNet/EfficientNet precisam de mais detalhes)
img_size = 96
batch_size = 64

# Data Augmentation + Normalização
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2  # separa 20% do treino para validação
)

# Gerador de treino
train_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(img_size, img_size),  # agora 96x96
    color_mode="grayscale",           # mantém grayscale (replicamos no modelo p/ RGB)
    class_mode="categorical",
    batch_size=batch_size,
    subset="training",
    shuffle=True
)

# Gerador de validação
val_generator = train_datagen.flow_from_directory(
    "data/train",
    target_size=(img_size, img_size),
    color_mode="grayscale",
    class_mode="categorical",
    batch_size=batch_size,
    subset="validation",
    shuffle=False
)

# Número de classes (usado depois no modelo)
num_classes = train_generator.num_classes
print(f"Número de classes detectadas: {num_classes}")



Found 22968 images belonging to 7 classes.
Found 5741 images belonging to 7 classes.
Número de classes detectadas: 7


In [15]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

# Replicamos 1 canal (grayscale) para 3 canais antes de passar na MobileNetV2
inputs = Input(shape=(96, 96, 1), name="input_gray")
x = Conv2D(3, (3, 3), padding="same", activation=None, name="gray_to_rgb")(inputs)
x = BatchNormalization()(x)

# Backbone MobileNetV2 pré-treinado no ImageNet
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(96, 96, 3)
)

# Congela inicialmente todas as camadas
for layer in base_model.layers:
    layer.trainable = False

# Libera fine-tuning das camadas acima do índice 50
for layer in base_model.layers[50:]:
    layer.trainable = True

# Conecta a entrada replicada (x) ao backbone
x = base_model(x, training=False)

# Pooling + cabeça de classificação
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
outputs = Dense(num_classes, activation="softmax")(x)

# Modelo final
model = Model(inputs, outputs, name="MobileNetV2_FER2013")

# Compilação
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),  # learning rate menor para fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "MobileNetV2_FER2013"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_gray (InputLayer)         │ (None, 96, 96, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gray_to_rgb (Conv2D)            │ (None, 96, 96, 3)      │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 96, 96, 3)      │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,993 (8.65 MB)

 Trainable params: 2,183,595 (8.33 MB)

 Non-trainable params: 83,398 (325.77 KB)

In [16]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ModelCheckpoint("mobilenetv2_fer2013.h5", monitor="val_loss", save_best_only=True, verbose=1)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=100,
    callbacks=callbacks
)


Epoch 1/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2762 - loss: 2.0876
Epoch 1: val_loss improved from inf to 1.88390, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 811s 2s/step - accuracy: 0.2764 - loss: 2.0870 - val_accuracy: 0.3681 - val_loss: 1.8839 - learning_rate: 1.0000e-04
Epoch 2/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4058 - loss: 1.5459
Epoch 2: val_loss improved from 1.88390 to 1.74995, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 744s 2s/step - accuracy: 0.4058 - loss: 1.5457 - val_accuracy: 0.4287 - val_loss: 1.7500 - learning_rate: 1.0000e-04
Epoch 3/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4812 - loss: 1.3703
Epoch 3: val_loss improved from 1.74995 to 1.62514, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 747s 2s/step - accuracy: 0.4812 - loss: 1.3703 - val_accuracy: 0.4703 - val_loss: 1.6251 - learning_rate: 1.0000e-04
Epoch 4/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5024 - loss: 1.2978
Epoch 4: val_loss improved from 1.62514 to 1.52868, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 664s 2s/step - accuracy: 0.5024 - loss: 1.2977 - val_accuracy: 0.4868 - val_loss: 1.5287 - learning_rate: 1.0000e-04
Epoch 5/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5300 - loss: 1.2406
Epoch 5: val_loss improved from 1.52868 to 1.36494, saving model to mobilenetv2_fer2013.h5


...

359/359 ━━━━━━━━━━━━━━━━━━━━ 665s 2s/step - accuracy: 0.5897 - loss: 1.0821 - val_accuracy: 0.5562 - val_loss: 1.1593 - learning_rate: 1.0000e-04
Epoch 10/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5972 - loss: 1.0726
Epoch 10: val_loss did not improve from 1.15925
359/359 ━━━━━━━━━━━━━━━━━━━━ 692s 2s/step - accuracy: 0.5972 - loss: 1.0726 - val_accuracy: 0.5654 - val_loss: 1.1866 - learning_rate: 1.0000e-04
Epoch 11/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6123 - loss: 1.0310
Epoch 11: val_loss did not improve from 1.15925
359/359 ━━━━━━━━━━━━━━━━━━━━ 666s 2s/step - accuracy: 0.6123 - loss: 1.0311 - val_accuracy: 0.5614 - val_loss: 1.1772 - learning_rate: 1.0000e-04
Epoch 12/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6098 - loss: 1.0239
Epoch 12: val_loss improved from 1.15925 to 1.13062, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 690s 2s/step - accuracy: 0.6098 - loss: 1.0239 - val_accuracy: 0.5743 - val_loss: 1.1306 - learning_rate: 1.0000e-04
Epoch 13/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6159 - loss: 1.0042
Epoch 13: val_loss did not improve from 1.13062
359/359 ━━━━━━━━━━━━━━━━━━━━ 662s 2s/step - accuracy: 0.6159 - loss: 1.0041 - val_accuracy: 0.5804 - val_loss: 1.1341 - learning_rate: 1.0000e-04
Epoch 14/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6348 - loss: 0.9819
Epoch 14: val_loss improved from 1.13062 to 1.09838, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 694s 2s/step - accuracy: 0.6348 - loss: 0.9819 - val_accuracy: 0.5835 - val_loss: 1.0984 - learning_rate: 1.0000e-04
Epoch 15/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6345 - loss: 0.9657
Epoch 15: val_loss did not improve from 1.09838
359/359 ━━━━━━━━━━━━━━━━━━━━ 700s 2s/step - accuracy: 0.6345 - loss: 0.9657 - val_accuracy: 0.5752 - val_loss: 1.1242 - learning_rate: 1.0000e-04
Epoch 16/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6509 - loss: 0.9322
Epoch 16: val_loss improved from 1.09838 to 1.08918, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 651s 2s/step - accuracy: 0.6509 - loss: 0.9322 - val_accuracy: 0.5955 - val_loss: 1.0892 - learning_rate: 1.0000e-04
Epoch 17/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6493 - loss: 0.9266
Epoch 17: val_loss did not improve from 1.08918
359/359 ━━━━━━━━━━━━━━━━━━━━ 650s 2s/step - accuracy: 0.6494 - loss: 0.9266 - val_accuracy: 0.5954 - val_loss: 1.0926 - learning_rate: 1.0000e-04
Epoch 18/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6625 - loss: 0.8975
Epoch 18: val_loss improved from 1.08918 to 1.06259, saving model to mobilenetv2_fer2013.h5


359/359 ━━━━━━━━━━━━━━━━━━━━ 682s 2s/step - accuracy: 0.6624 - loss: 0.8976 - val_accuracy: 0.6001 - val_loss: 1.0626 - learning_rate: 1.0000e-04
Epoch 19/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6614 - loss: 0.8948
Epoch 19: val_loss did not improve from 1.06259
359/359 ━━━━━━━━━━━━━━━━━━━━ 652s 2s/step - accuracy: 0.6614 - loss: 0.8948 - val_accuracy: 0.5804 - val_loss: 1.1113 - learning_rate: 1.0000e-04
Epoch 20/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6721 - loss: 0.8707
Epoch 20: val_loss did not improve from 1.06259
359/359 ━━━━━━━━━━━━━━━━━━━━ 653s 2s/step - accuracy: 0.6721 - loss: 0.8707 - val_accuracy: 0.5830 - val_loss: 1.1352 - learning_rate: 1.0000e-04
Epoch 21/100
359/359 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6875 - loss: 0.8289
Epoch 21: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 21: val_loss did not improve from 1.06259
359/359 ━━━━━━━━━━━━━━━━━━━━ 654s 2s/step - accuracy: 0.6875 - loss: 0.8290 - v

In [17]:
from tensorflow.keras.models import load_model
model = load_model("mobilenetv2_fer2013.h5")


In [18]:
val_loss, val_acc = model.evaluate(val_generator)
print(f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")


90/90 ━━━━━━━━━━━━━━━━━━━━ 56s 585ms/step - accuracy: 0.5487 - loss: 1.2011
Val Loss: 1.0798, Val Accuracy: 0.6030


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam

# Carrega o melhor modelo salvo
model = load_model("mobilenetv2_fer2013.h5")

# Recompila com o LR ajustado pelo ReduceLROnPlateau
model.compile(
    optimizer=Adam(learning_rate=2.5e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Continua a partir da epoch 25
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,              # limite final
    initial_epoch=25,        # continua de onde parou
    callbacks=callbacks,
    verbose=1
)



Epoch 26/50
179/359 ━━━━━━━━━━━━━━━━━━━━ 5:00 2s/step - accuracy: 0.6841 - loss: 0.8488